# v6 — Interaction features

**What v6 adds.** Three engineered features on top of v5's stack (`geo_cluster` k=30, cleaned `funder`/`installer`):
- `quantity_x_waterpoint_type` — string concatenation, 32 observed levels
- `payment_x_management` — string concatenation, 76 observed levels
- `population_density_ratio` — `population` ÷ the region's mean population

Chosen because "DrivenData forum discussions flagged them". Balanced weights stay in the shipped version.

**Hypotheses.**
- **H7c** — explicit interactions sharpen splits at limited depth. *Counter-argument:* with depth-12 trees (the v3 search winner) a tree can already form these interactions itself, so the gain should be near zero.
- **H5⁗** — weights cost accuracy; main protocol is **no weights**.

**Protocol.** v3's searched params, 300 trees, no weights, same folds, features fit per fold. Reference = v5 row **5b** (`geo_cluster` + cleaned text, no interactions) **0.8115**. Each interaction is tested alone and together, since a combined +0 can hide one helping and one hurting.

| row | change vs reference |
|---|---|
| ref | v5 row 5b (read from `results.jsonl`) |
| 6a | all three interactions (**v6's features**) |
| 6b | only `quantity_x_waterpoint_type` |
| 6c | only `payment_x_management` |
| 6d | only `population_density_ratio` |
| 6e | **v6 as shipped**: 6a + early stopping on val fold + **weights** |

In [1]:
import sys, warnings, json
sys.path.insert(0, '.')
warnings.simplefilter('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import *
pd.set_option('display.width', 140)
df, y, le = get_train('v6')
Flex = make_flex('v6')
BEST = json.loads(open('cache/v3_search.json').read())['best_params']
ref = next(r for r in map(json.loads, RESULTS.read_text().splitlines()) if r['label'].startswith('5b '))
print(f"reference 5b (v5 features, no interactions, no weights): {ref['mean']:.4f} ± {ref['std']:.4f}")
BASE = dict(k=30, text=dict(cols=['funder', 'installer'], min=20))

def run(label, inter, **kw):
    sc, oof, it = cv_xgb_es(df, y, lambda: Flex(**BASE, inter=inter), BEST, **kw)
    r = log_result('v6', label, sc)
    d = np.array(sc) - np.array(ref['folds'])
    print(f"{label:<44} {r['mean']:.4f} ± {r['std']:.4f}   Δ vs ref {d.mean():+.4f}  per-fold {np.round(d, 4).tolist()}")
    return sc, oof

reference 5b (v5 features, no interactions, no weights): 0.8115 ± 0.0036


## Do the interactions carry anything the parts don't? (before any model)

In [2]:
from sklearn.metrics import adjusted_mutual_info_score as ami
X = df.drop(columns=[TARGET])
qw = X.quantity.astype(str) + '__' + X.waterpoint_type.astype(str)
pm = X.payment.astype(str) + '__' + X.management.astype(str)
rows = [('quantity', ami(X.quantity, df[TARGET])), ('waterpoint_type', ami(X.waterpoint_type, df[TARGET])), ('quantity × waterpoint_type', ami(qw, df[TARGET])),
        ('payment', ami(X.payment, df[TARGET])), ('management', ami(X.management, df[TARGET])), ('payment × management', ami(pm, df[TARGET]))]
print('AMI with target (finer partitions score higher even by chance, so read relative to the parts):')
pd.DataFrame(rows, columns=['feature', 'AMI with target']).set_index('feature').round(4)

AMI with target (finer partitions score higher even by chance, so read relative to the parts):


,AMI with target
feature,
quantity,0.1025
waterpoint_type,0.0596
quantity × waterpoint_type,0.0989
payment,0.0273
management,0.0171
payment × management,0.0315


## 6a — all three interactions

In [3]:
sc_a, oof_a = run('6a all three interactions (v6 features)', ('qw', 'pm', 'pop'), es='none')

6a all three interactions (v6 features)      0.8108 ± 0.0039   Δ vs ref -0.0007  per-fold [-0.0008, 0.0015, -0.0003, -0.0012, -0.0026]


## 6b / 6c / 6d — each interaction alone

In [4]:
sc_b, _ = run('6b quantity x waterpoint_type only', ('qw',), es='none')
sc_c, _ = run('6c payment x management only', ('pm',), es='none')
sc_d, _ = run('6d population_density_ratio only', ('pop',), es='none')

6b quantity x waterpoint_type only           0.8116 ± 0.0041   Δ vs ref +0.0001  per-fold [0.0008, 0.0016, -0.001, 0.0, -0.0009]


6c payment x management only                 0.8106 ± 0.0035   Δ vs ref -0.0010  per-fold [-0.0016, -0.0007, -0.0003, -0.0013, -0.0011]


6d population_density_ratio only             0.8113 ± 0.0046   Δ vs ref -0.0002  per-fold [0.0017, 0.0006, -0.0008, -0.0012, -0.0013]


## 6e — v6 as shipped (interactions + early stopping on val fold + balanced weights)

In [5]:
sc_e, oof_e = run('6e v6 as shipped (val ES + weights)', ('qw', 'pm', 'pop'), es='val', balanced=True)

6e v6 as shipped (val ES + weights)          0.7817 ± 0.0056   Δ vs ref -0.0298  per-fold [-0.0254, -0.0325, -0.0301, -0.0305, -0.0308]


## Does the model use them?

In [6]:
import xgboost as xgb
Xf = Flex(**BASE, inter=('qw', 'pm', 'pop')).fit_transform(X)
clf = xgb.XGBClassifier(**BEST, n_estimators=300, tree_method='hist', enable_categorical=True, random_state=SEED, n_jobs=-1).fit(Xf, y)
imp = pd.Series(clf.get_booster().get_score(importance_type='total_gain')).sort_values(ascending=False)
imp = (imp / imp.sum()).round(4)
for c in ['quantity_x_waterpoint_type', 'payment_x_management', 'population_density_ratio']:
    print(f'{c:<30} rank {list(imp.index).index(c) + 1:>2} of {len(imp)}, gain share {imp[c]:.4f}')
imp.head(8)

quantity_x_waterpoint_type     rank  1 of 45, gain share 0.1847
payment_x_management           rank  6 of 45, gain share 0.0450
population_density_ratio       rank  8 of 45, gain share 0.0283


quantity_x_waterpoint_type    0.1847
ward                          0.1519
lga                           0.1181
funder                        0.0558
installer                     0.0500
payment_x_management          0.0450
quantity                      0.0437
population_density_ratio      0.0283
dtype: float64

## Results

In [7]:
res = pd.DataFrame([r for r in map(json.loads, RESULTS.read_text().splitlines()) if r['version'] == 'v6'])
res['acc'] = res['mean'].round(4); res['fold std'] = res['std'].round(4)
res['Δ vs ref'] = (res['mean'] - ref['mean']).round(4)
res[['label', 'acc', 'fold std', 'Δ vs ref']].set_index('label')

,acc,fold std,Δ vs ref
label,,,
6a all three interactions (v6 features),0.8108,0.0039,-0.0007
6b quantity x waterpoint_type only,0.8116,0.0041,0.0001
6c payment x management only,0.8106,0.0035,-0.0010
6d population_density_ratio only,0.8113,0.0046,-0.0002
6e v6 as shipped (val ES + weights),0.7817,0.0056,-0.0298


## Conclusion

Searched params, 300 trees, same folds, reference = v5 row 5b (`geo_cluster` + cleaned text, no interactions, no weights) **0.8115 ± 0.0036**.

| row | CV accuracy | Δ vs ref |
|---|---|---|
| 6a all three interactions (**v6's features**) | 0.8108 ± 0.0039 | −0.0007 |
| 6b `quantity × waterpoint_type` only | 0.8116 ± 0.0041 | +0.0001 |
| 6c `payment × management` only | 0.8106 ± 0.0035 | −0.0010 |
| 6d `population_density_ratio` only | 0.8113 ± 0.0046 | −0.0002 |
| **6e v6 as shipped (val ES + weights)** | **0.7817 ± 0.0056** | **−0.0298** |

- **H7c (explicit interactions help) is not supported.** Individually and together the interactions are within ±0.001 of the reference, a quarter of the fold std or less. If anything the combined set is very slightly *worse* (−0.0007), and `payment × management` is negative on all 5 folds (−0.0003 to −0.0016), small but consistent. Depth-12 trees can build these combinations themselves, which is the counter-argument stated up front.
- **Importance is not lift.** The model ranks `quantity_x_waterpoint_type` **1st of 45** by gain (18% share), `payment_x_management` 6th and `population_density_ratio` 8th, yet none improves accuracy. The interaction simply takes over gain that `quantity` and `waterpoint_type` would otherwise share. A gain ranking says which column the trees *chose*, not whether the model got better.
- **The pre-model check agrees.** AMI with the target: `quantity` 0.1025, `waterpoint_type` 0.0596, but their product only 0.0989, so the combination carries nothing that `quantity` alone doesn't. `payment × management` (0.0315) beats `payment` (0.0273) only slightly.
- **Weights, a fifth time.** v6 as shipped scores 0.7817, the same as v3 / v4 / v5 (0.7813–0.7818). Four versions in a row changed the shipped score by under 0.001.

**Caveats.** One seed, v3's params reused (v6 re-searched), CV only. Interactions might help a shallower model or a linear one. I only tested depth-12 XGBoost.

**Running total (v1–v6).** The honest best is still ≈ 0.811, set by v3's tuned params, and has not moved since. Everything after v3 (cluster, text cleaning, interactions) is within noise. The changes that did beat noise were all *subtractions or corrections*: drop the top-15 cut (+0.006), drop class weights (+0.03), keep `funder`/`installer` (+0.002).

Next: **v7**, the XGBoost + LightGBM blend, the last version.